In [1]:
!pip install -U pip setuptools==68.0.0 wheel

!pip install \
    numpy==1.24.3 \
    pandas==2.0.3 \
    pytorch_lightning==2.0.6 \
    scikit_learn==1.3.1 \
    torch==2.0.1 \
    tqdm==4.65.0 \
    transformers==4.33.3 \
    peft==0.5.0 \
    accelerate==0.23.0

!pip install microformer-mgm
!pip install optuna
!pip uninstall -y jax jaxlib

In [2]:
import os
import torch
import optuna
import pandas as pd
import numpy as np

from pickle import load, dump

from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import roc_auc_score

from transformers import (
    GPT2ForSequenceClassification,
    Trainer,
    TrainingArguments
)

from sklearn.model_selection import train_test_split
from transformers.trainer_callback import EarlyStoppingCallback

from mgm.src.MicroCorpus import SequenceClassificationDataset

In [3]:
import zipfile

with zipfile.ZipFile('MGM_NHANES_DAPT.zip', 'r') as zip_ref:
    zip_ref.extractall('MGM_NHANES_DAPT')

In [4]:
!mgm construct \
-i asv_genus.csv \
-o corpus.pkl

Starting MGM...
No config file provided, use default config file.
No seed provided, the program will generate a random seed.
Your data will be normalized with the phylogeny mean and std. If you wish to use your own normalization, please use --no-normalize.
0 samples are dropped for all zeroes
100% 101/101 [00:00<00:00, 2168.65it/s]
Total 101 samples.
            Max length is 80.
            Average length is 66.00990099009901.
            Min length is 47.


In [5]:
from pickle import load

corpus = load(open("corpus.pkl","rb"))

corpus[0]

{'input_ids': tensor([   2, 7158, 3768, 5803,  160, 9429, 8582, 8597, 8169, 4649,  364,  242,
         8328, 3584, 3840, 5251, 1624, 1070, 1677, 1695,    9, 7858, 4831, 3013,
         2384, 8777, 4545, 8050,  486,  367, 1304,  851, 6475, 1746, 6778, 7443,
         7214,  148, 4491, 6196, 7120, 6122, 4721, 5685, 8219, 3481, 9183, 1421,
         6640, 2300, 5572, 6638, 3516, 2707, 2736, 8495, 1095, 4666,    3,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
            0, 

In [6]:
df_metadata = pd.read_csv(
    "df_metadata.csv"
)


df_metadata.head()

,Unnamed: 0,Run,disease_status,pair_id
0,0,SRR32936395,1,17
1,1,SRR32936396,0,16
2,2,SRR32936397,1,3
3,3,SRR32936398,1,16
4,4,SRR32936408,1,2


In [7]:
cv = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)


folds = list(
    cv.split(
        df_metadata,
        df_metadata["disease_status"],
        groups=df_metadata["pair_id"]
    )
)


print(len(folds))

5


In [8]:
def set_trainable_layers(
    model,
    mode
):

    for p in model.parameters():
        p.requires_grad=False


    if mode=="all":

        for p in model.parameters():
            p.requires_grad=True


    elif mode=="last2":

        for layer in model.transformer.h[-2:]:
            for p in layer.parameters():
                p.requires_grad=True


    elif mode=="last4":

        for layer in model.transformer.h[-4:]:
            for p in layer.parameters():
                p.requires_grad=True


    for p in model.score.parameters():
        p.requires_grad=True

In [9]:
def compute_metrics(eval_pred):

    logits, labels = eval_pred

    probs = torch.softmax(
        torch.tensor(logits),
        dim=1
    )[:, 1].numpy()

    auc = roc_auc_score(
        labels,
        probs
    )

    return {
        "auc": auc
    }

In [10]:
def train_fold(
    train_idx,
    val_idx,
    params,
    fold
):


    train_meta = df_metadata.iloc[train_idx]

    val_meta = df_metadata.iloc[val_idx]


    train_ids = train_meta["Run"].values
    val_ids = val_meta["Run"].values



    train_labels = (
        train_meta
        .set_index("Run")["disease_status"]
    )


    val_labels = (
        val_meta
        .set_index("Run")["disease_status"]
    )


    train_positions = [
        corpus.data.index.get_loc(x)
        for x in train_ids
    ]


    val_positions = [
        corpus.data.index.get_loc(x)
        for x in val_ids
    ]



    train_dataset = SequenceClassificationDataset(
        corpus[train_positions]["input_ids"],
        corpus[train_positions]["attention_mask"],
        torch.tensor(train_labels.values)
    )


    val_dataset = SequenceClassificationDataset(
        corpus[val_positions]["input_ids"],
        corpus[val_positions]["attention_mask"],
        torch.tensor(val_labels.values)
    )


    # cargar DAPT
    model = GPT2ForSequenceClassification.from_pretrained(
        "MGM_NHANES_DAPT",
        num_labels=2
    )


    set_trainable_layers(
        model,
        params["layers"]
    )
    args = TrainingArguments(

        output_dir=f"optuna_fold_{fold}",

        learning_rate=params["learning_rate"],

        weight_decay=params["weight_decay"],

        warmup_ratio=params["warmup_ratio"],

        num_train_epochs=20,

        per_device_train_batch_size=8,

        evaluation_strategy="epoch",

        save_strategy="epoch",

        load_best_model_at_end=True,

        metric_for_best_model="eval_auc",

        greater_is_better=True,

        logging_strategy="epoch",

        report_to="none"
    )

    trainer = Trainer(

        model=model,

        args=args,

        train_dataset=train_dataset,

        eval_dataset=val_dataset,

        compute_metrics=compute_metrics,

        callbacks=[
            EarlyStoppingCallback(
                early_stopping_patience=3
            )
        ]
    )

    trainer.train()


    preds = trainer.predict(
        val_dataset
    )


    probs = torch.softmax(
        torch.tensor(preds.predictions),
        dim=1
    )[:,1]


    auc = roc_auc_score(
        val_labels.values,
        probs.detach().numpy()
    )

    del model
    del trainer

    torch.cuda.empty_cache()

    return auc

In [11]:
def evaluate_params(params):

    aucs = []

    for fold, (train_idx, val_idx) in enumerate(folds):

        print(f"\nFold {fold+1}")

        auc = train_fold(
            train_idx=train_idx,
            val_idx=val_idx,
            params=params,
            fold=fold
        )

        print(f"AUC = {auc:.4f}")

        aucs.append(auc)

    print("\n====================")
    print("Mean AUC:", np.mean(aucs))
    print("Std AUC :", np.std(aucs))

    return aucs

In [12]:
def objective(trial):

    params = {

        "learning_rate":
        trial.suggest_float(
            "learning_rate",
            1e-5,
            5e-4,
            log=True
        ),

        "weight_decay":
        trial.suggest_categorical(
            "weight_decay",
            [0,0.01,0.1]
        ),

        "warmup_ratio":
        trial.suggest_categorical(
            "warmup_ratio",
            [0,0.1,0.2]
        ),

        "layers":
        trial.suggest_categorical(
            "layers",
            ["all","last4","last2"]
        )
    }


    try:

        scores=[]

        for fold,(train_idx,val_idx) in enumerate(folds):

            auc=train_fold(
                train_idx,
                val_idx,
                params,
                fold
            )

            scores.append(auc)


        return np.mean(scores)


    except torch.cuda.OutOfMemoryError:

        torch.cuda.empty_cache()

        return 0.0

In [13]:
study = optuna.create_study(
    direction="maximize"
)


study.optimize(
    objective,
    n_trials=30
)

'\nstudy = optuna.create_study(\n    direction="maximize"\n)\n\n\nstudy.optimize(\n    objective,\n    n_trials=30\n)\n'

In [ ]:
study.best_params

In [ ]:
pd.DataFrame(
    study.trials_dataframe()
).to_csv(
    "optuna_results.csv",
    index=False
)

In [15]:
params_last4 = {
    "learning_rate": 3.3410221978461924e-4,
    "weight_decay": 0.01,
    "warmup_ratio": 0.2,
    "layers": "last4"
}

auc_last4 = evaluate_params(params_last4)

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at MGM_NHANES_DAPT and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Fold 1


Epoch,Training Loss,Validation Loss,Auc
1,0.697600,0.710784,0.393939
2,0.691000,0.695014,0.446970
3,0.670600,0.688208,0.583333
4,0.655100,0.670384,0.712121
5,0.613300,0.668987,0.712121
6,0.547500,0.631946,0.689394
7,0.441900,0.777923,0.674242


AUC = 0.7121

Fold 2


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at MGM_NHANES_DAPT and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


...

Epoch,Training Loss,Validation Loss,Auc
1,0.704700,0.709367,0.512500
2,0.688000,0.679347,0.612500
3,0.679500,0.668505,0.675000
4,0.647700,0.658567,0.675000
5,0.612500,0.625664,0.700000
6,0.529900,0.600602,0.712500
7,0.433900,0.622857,0.725000
8,0.334200,0.667648,0.725000
9,0.276000,0.791027,0.762500
10,0.279300,0.810740,0.737500


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at MGM_NHANES_DAPT and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


AUC = 0.7750

Fold 5


Epoch,Training Loss,Validation Loss,Auc
1,0.705000,0.706442,0.450000
2,0.690200,0.701354,0.450000
3,0.677000,0.713898,0.430000
4,0.626100,0.726268,0.480000
5,0.585300,0.758437,0.470000
6,0.448100,0.936612,0.480000
7,0.355400,0.955799,0.540000
8,0.298100,1.130108,0.540000
9,0.193300,1.335059,0.500000
10,0.126900,1.503391,0.480000


AUC = 0.5400

Mean AUC: 0.7054242424242425
Std AUC : 0.13223071636643483


In [16]:
params_all = {
    "learning_rate": 4.59324400326387e-4,
    "weight_decay": 0.1,
    "warmup_ratio": 0.1,
    "layers": "all"
}

auc_all = evaluate_params(params_all)

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at MGM_NHANES_DAPT and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



Fold 1


Epoch,Training Loss,Validation Loss,Auc
1,0.707800,0.696673,0.492424
2,0.678000,0.734609,0.659091
3,0.585900,0.673591,0.750000
4,0.541100,0.984818,0.696970
5,0.361100,0.777136,0.704545
6,0.135700,1.070987,0.689394


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at MGM_NHANES_DAPT and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


AUC = 0.7500

Fold 2


...

Epoch,Training Loss,Validation Loss,Auc
1,0.700200,0.675108,0.650000
2,0.659600,0.616371,0.725000
3,0.526000,0.655880,0.712500
4,0.363200,0.772230,0.700000
5,0.185700,0.857356,0.725000


Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at MGM_NHANES_DAPT and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


AUC = 0.7250

Fold 5


Epoch,Training Loss,Validation Loss,Auc
1,0.690800,0.698490,0.510000
2,0.653000,0.756775,0.480000
3,0.502900,0.882875,0.520000
4,0.328400,0.999139,0.550000
5,0.141400,1.665960,0.540000
6,0.047500,2.366354,0.590000
7,0.036200,2.402586,0.460000
8,0.178900,1.947848,0.530000
9,0.021300,2.708033,0.530000


AUC = 0.5900

Mean AUC: 0.7070000000000001
Std AUC : 0.10380751417888788


In [17]:
print("\nRESULTADOS")

print(f"last4: {np.mean(auc_last4):.4f} ± {np.std(auc_last4):.4f}")
print(f"all  : {np.mean(auc_all):.4f} ± {np.std(auc_all):.4f}")


RESULTADOS
last4: 0.7054 ± 0.1322
all  : 0.7070 ± 0.1038


In [18]:
# eliminar muestras de metadata que no existen en el corpus
df_final = df_metadata[
    df_metadata["Run"].isin(corpus.data.index)
].copy()


print("Muestras finales:", len(df_final))
print(df_final["disease_status"].value_counts())

Muestras finales: 101
disease_status
1    51
0    50
Name: count, dtype: int64


In [19]:
final_positions = [
    corpus.data.index.get_loc(x)
    for x in df_final["Run"]
]

In [20]:
dataset_final = SequenceClassificationDataset(

    corpus[final_positions]["input_ids"],

    corpus[final_positions]["attention_mask"],

    torch.tensor(
        df_final["disease_status"].values,
        dtype=torch.long
    )
)

In [21]:
print(len(dataset_final))
print(dataset_final[0])

101
{'input_ids': tensor([   2, 8050, 8169, 1291, 7158, 4545,  486, 9429, 7120, 3013, 8597,  160,
        1677,  364, 8777, 3768, 3584, 2300, 2384,  242, 8582, 9183, 4831,  367,
        1746, 2736, 6475, 1695, 1775, 1070, 7858, 4666, 1421, 8328, 5251, 6778,
        3481,  851, 7443, 6122, 4649, 4771,  148, 2667, 5685, 4227, 3840, 8292,
        3421, 8219, 3516, 1624, 6543, 6638, 6196, 1304, 5572,    9, 1095, 2707,
        8495, 5803, 7829,    3,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,    0,
           0,    0,   

In [22]:
train_idx, val_idx = train_test_split(
    np.arange(len(df_final)),
    test_size=0.20,
    stratify=df_final["disease_status"],
    random_state=42
)

df_train = df_final.iloc[train_idx].copy()
df_val = df_final.iloc[val_idx].copy()

In [23]:
print("Muestras de entrenamiento:", len(df_train))
print("Muestras de validación:", len(df_val))

print("\nDistribución entrenamiento:")
print(df_train["disease_status"].value_counts(normalize=True))

print("\nDistribución validación:")
print(df_val["disease_status"].value_counts(normalize=True))

Muestras de entrenamiento: 80
Muestras de validación: 21

Distribución entrenamiento:
disease_status
0    0.5
1    0.5
Name: proportion, dtype: float64

Distribución validación:
disease_status
1    0.52381
0    0.47619
Name: proportion, dtype: float64


In [24]:
train_positions = [
    corpus.data.index.get_loc(x)
    for x in df_train["Run"]
]

val_positions = [
    corpus.data.index.get_loc(x)
    for x in df_val["Run"]
]

In [25]:
train_dataset = SequenceClassificationDataset(
    corpus[train_positions]["input_ids"],
    corpus[train_positions]["attention_mask"],
    torch.tensor(
        df_train["disease_status"].values,
        dtype=torch.long
    )
)

In [26]:
val_dataset = SequenceClassificationDataset(
    corpus[val_positions]["input_ids"],
    corpus[val_positions]["attention_mask"],
    torch.tensor(
        df_val["disease_status"].values,
        dtype=torch.long
    )
)

In [27]:
print("Muestras de entrenamiento:", len(train_dataset))
print("Muestras de validación:", len(val_dataset))

Muestras de entrenamiento: 80
Muestras de validación: 21


In [28]:
model_final = GPT2ForSequenceClassification.from_pretrained(
    "MGM_NHANES_DAPT",
    num_labels=2
)

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at MGM_NHANES_DAPT and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [29]:
best_params = {
    "learning_rate": 4.59324400326387e-4,
    "weight_decay": 0.1,
    "warmup_ratio": 0.1
}

In [30]:
print("Learning rate:", best_params["learning_rate"])
print("Weight decay:", best_params["weight_decay"])
print("Warmup ratio:", best_params["warmup_ratio"])

Learning rate: 0.000459324400326387
Weight decay: 0.1
Warmup ratio: 0.1


In [31]:
training_args_final = TrainingArguments(
    output_dir="MGM_NHANES_finetuned_all",

    learning_rate=best_params["learning_rate"],

    weight_decay=best_params["weight_decay"],

    warmup_ratio=best_params["warmup_ratio"],

    num_train_epochs=50,

    per_device_train_batch_size=8,

    evaluation_strategy="epoch",

    save_strategy="epoch",

    load_best_model_at_end=True,

    metric_for_best_model="eval_loss",

    greater_is_better=False,

    logging_strategy="epoch",

    report_to="none"
)

In [32]:
trainer_final = Trainer(
    model=model_final,

    args=training_args_final,

    train_dataset=train_dataset,

    eval_dataset=val_dataset,

    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=5
        )
    ]
)

In [33]:
trainer_final.train()

Epoch,Training Loss,Validation Loss
1,0.701000,0.724599
2,0.692800,0.710664
3,0.654300,0.703136
4,0.599300,0.682244
5,0.505700,0.641506
6,0.371500,0.582650
7,0.215300,0.999103
8,0.036900,0.766958
9,0.100400,0.866252
10,0.188400,2.288421


TrainOutput(global_step=110, training_loss=0.3748581886291504, metrics={'train_runtime': 20.4033, 'train_samples_per_second': 196.046, 'train_steps_per_second': 24.506, 'total_flos': 17082812989440.0, 'train_loss': 0.3748581886291504, 'epoch': 11.0})

In [34]:
trainer_final.save_model(
    "MGM_NHANES_finetuned_all"
)